# ⚖️ Contract Intelligence Engine — FAST VERSION
### Hireathon 2026 — Problem Statement 3

> Models already trained. This notebook loads them and launches the UI.
> **Optimized for speed — typical contract analyzes in 20–40 seconds.**

| Cell | What it does |
|------|-------------|
| 1 | Install gradio + pymupdf |
| 2 | ⚠️ Restart Runtime |
| 3 | All imports |
| 4 | Load saved models |
| 5 | Clause questions (15 types) |
| 6 | Rule-based extractors |
| 7 | QA extractor with confidence filter |
| 8 | Fast chunker |
| 9 | Fast batch analyzer |
| 10 | Risk engine |
| 11 | PDF extractor |
| 12 | 🚀 Launch Gradio UI |

In [1]:
!pip uninstall -y transformers peft accelerate -q

!pip install -q \
transformers==4.41.2 \
peft==0.10.0 \
accelerate==0.30.1 \
datasets \
pandas \
scikit-learn \
pymupdf \
ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 87.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.1 MB/s eta 0:00:0000:01


In [2]:
import os, json, torch, fitz, re
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)

import ipywidgets as widgets
from IPython.display import display

2026-03-28 09:53:57.016086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774691637.176666     301 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774691637.222497     301 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774691637.565885     301 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774691637.565920     301 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774691637.565922     301 computation_placer.cc:177] computation placer alr

In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for f in files[:3]:
        print("   ", f)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ritviksingh12
/kaggle/input/datasets/ritviksingh12/cuadv1
    CUADv1.json
    train_separate_questions.json
    test.json
/kaggle/input/datasets/ritviksingh12/contracts
    contract4.pdf
    contract5.pdf
    Contract1.pdf


In [8]:
def process_cuad_multilabel(path):
    with open(path) as f:
        data = json.load(f)

    rows = []
    for doc in data["data"]:
        for para in doc["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                rows.append({
                    "text": context[:512],
                    "question": qa["question"],
                    "label": 0 if qa["is_impossible"] else 1
                })

    df = pd.DataFrame(rows)

    pivot = df.pivot_table(
        index="text",
        columns="question",
        values="label",
        fill_value=0
    ).reset_index()

    os.makedirs("data", exist_ok=True)
    pivot.to_csv("data/train_multi.csv", index=False)

process_cuad_multilabel("/kaggle/input/datasets/ritviksingh12/cuadv1/CUADv1.json")

In [9]:
df = pd.read_csv("data/train_multi.csv")

def clean(col):
    return col.split('"')[1] if '"' in col else col

df.columns = ["text"] + [clean(c) for c in df.columns[1:]]
df.to_csv("data/train_multi_clean.csv", index=False)

In [10]:
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

df = pd.read_csv("data/train_multi_clean.csv")
labels = df.columns[1:]

df["labels"] = df[labels].values.tolist()
dataset = Dataset.from_pandas(df[["text", "labels"]])

def tokenize(x):
    return tokenizer(x["text"], truncation=True, padding="max_length", max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.1)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    problem_type="multi_label_classification"
)

training_args = TrainingArguments(
    output_dir="./results_multi",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    fp16=True,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=dataset["train"],
                  eval_dataset=dataset["test"])

trainer.train()

model.save_pretrained("final_model_multi")
tokenizer.save_pretrained("final_model_multi")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/508 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss


('final_model_multi/tokenizer_config.json',
 'final_model_multi/special_tokens_map.json',
 'final_model_multi/vocab.txt',
 'final_model_multi/added_tokens.json',
 'final_model_multi/tokenizer.json')

In [16]:
def process_cuad_qa(path):
    with open(path) as f:
        data = json.load(f)

    rows = []
    for doc in data["data"]:
        for para in doc["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                if qa["is_impossible"] or not qa["answers"]:
                    continue

                rows.append({
                    "context": context,
                    "question": qa["question"],
                    "answer_text": qa["answers"][0]["text"],
                    "answer_start": qa["answers"][0]["answer_start"]
                })

    pd.DataFrame(rows).to_csv("data/train_qa.csv", index=False)

process_cuad_qa("/kaggle/input/datasets/ritviksingh12/cuadv1/CUADv1.json")

In [15]:
model_name = "deepset/bert-base-cased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

df = pd.read_csv("data/train_qa.csv")
dataset = Dataset.from_pandas(df)

def preprocess(example):
    inputs = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        padding="max_length",
        max_length=256,
        return_offsets_mapping=True
    )

    offsets = inputs["offset_mapping"]
    start = example["answer_start"]
    end = start + len(example["answer_text"])

    seq_ids = inputs.sequence_ids()

    s, e = 0, 0
    for i, (o, sid) in enumerate(zip(offsets, seq_ids)):
        if sid != 1:
            continue
        if o[0] <= start <= o[1]:
            s = i
        if o[0] <= end <= o[1]:
            e = i

    inputs["start_positions"] = s
    inputs["end_positions"] = e
    return inputs

dataset = dataset.map(preprocess)
dataset = dataset.train_test_split(test_size=0.1)

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./qa_results",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    fp16=True,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=dataset["train"],
                  eval_dataset=dataset["test"])

trainer.train()

model.save_pretrained("qa_model")
tokenizer.save_pretrained("qa_model")

Map:   0%|          | 0/6702 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/bert-base-cased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning

Step,Training Loss


('qa_model/tokenizer_config.json',
 'qa_model/special_tokens_map.json',
 'qa_model/vocab.txt',
 'qa_model/added_tokens.json',
 'qa_model/tokenizer.json')

In [1]:
# CELL 1 — Install only what's missing
!pip install -q gradio pymupdf
print('✅ Done — RESTART RUNTIME now, then run from Cell 3 downward')

✅ Done — RESTART RUNTIME now, then run from Cell 3 downward


## ⚠️ CELL 2 — RESTART RUNTIME
**Runtime → Restart Session** — then continue from Cell 3.

In [17]:
# CELL 3 — Imports
import os, re, torch, fitz
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering
)
import gradio as gr
print('✅ Imports OK')

✅ Imports OK


In [18]:
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForQuestionAnswering

# ✅ Use RELATIVE paths (CRITICAL FIX)
CLF_PATH = "final_model_multi"
QA_PATH  = "qa_model"
DATA_CSV = "data/train_multi_clean.csv"

# 🔍 Debug: check files exist
print("Files in working dir:", os.listdir())

assert os.path.exists(CLF_PATH), "❌ final_model_multi not found"
assert os.path.exists(QA_PATH), "❌ qa_model not found"
assert os.path.exists(DATA_CSV), "❌ train_multi_clean.csv not found"

# 🚀 Load models (LOCAL ONLY)
print('Loading classifier...')
clf_tokenizer = AutoTokenizer.from_pretrained(CLF_PATH, local_files_only=True)
clf_model     = AutoModelForSequenceClassification.from_pretrained(CLF_PATH, local_files_only=True)

print('Loading QA model...')
qa_tokenizer  = AutoTokenizer.from_pretrained(QA_PATH, local_files_only=True)
qa_model      = AutoModelForQuestionAnswering.from_pretrained(QA_PATH, local_files_only=True)

# ⚡ Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clf_model.to(device).eval()
qa_model.to(device).eval()

# 📊 Labels
label_cols = pd.read_csv(DATA_CSV).columns[1:].tolist()

print(f'✅ Models loaded successfully on {device}')
print(f'📊 Total clause labels: {len(label_cols)}')

Files in working dir: ['.gradio', 'qa_model', 'qa_results', 'final_model_multi', '.virtual_documents', 'results_multi', 'data']
Loading classifier...
Loading QA model...
✅ Models loaded successfully on cuda
📊 Total clause labels: 41


In [19]:
# CELL 5 — 15 Clause Questions

clause_questions = {
    'Document Name':               'What is the title of this agreement?',
    'Parties':                     'This agreement is between which parties?',
    'Agreement Date':              'This agreement is dated when?',
    'Effective Date':              'When does this agreement become effective?',
    'Expiration Date':             'When does this agreement expire?',
    'Governing Law':               'This agreement is governed by which law?',
    'Cap On Liability':            'Is there a maximum cap on damages or liability?',
    'Uncapped Liability':          'Is liability stated to be unlimited?',
    'Termination for Convenience': 'Can either party terminate without cause?',
    'Change Of Control':           'What happens if ownership or control changes?',
    'Anti-Assignment':             'Can this agreement be assigned to another party?',
    'Audit Rights':                'Does either party have audit rights?',
    'Exclusivity':                 'Is this agreement exclusive?',
    'Non-Compete':                 'Is competition or solicitation restricted?',
    'Revenue/Profit Sharing':      'Is revenue or profit shared between parties?'
}
print(f'✅ {len(clause_questions)} clause types defined')

✅ 15 clause types defined


In [20]:
# CELL 6 — Rule-based extractors (instant, no model)

def extract_parties_rule(text):
    patterns = [
        r'(?:between|BETWEEN)\s+(.+?)\s+(?:and|AND)\s+(.+?)(?:\s*[\.,\(])',
        r'(?:by and between)\s+(.+?)\s+(?:and)\s+(.+?)(?:\s*[\.,\(])',
    ]
    for pat in patterns:
        m = re.search(pat, text[:3000])
        if m:
            p1 = m.group(1).strip().strip('"').strip('(')
            p2 = m.group(2).strip().strip('"').strip('(')
            if 3 < len(p1) < 120 and 3 < len(p2) < 120:
                return f'{p1} and {p2}'
    return None

def extract_governing_law_rule(text):
    patterns = [
        r'governed by (?:the laws? of )?([A-Za-z][A-Za-z ,]+?)(?:\.|,|\s+and)',
        r'laws? of the [Ss]tate of ([A-Za-z ]+?)(?:\.|,)',
        r'jurisdiction of ([A-Za-z ]+?)(?:\.|,)',
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            result = m.group(1).strip()
            if 2 < len(result) < 80:
                return result
    return None

def extract_doc_name_rule(text):
    lines = [l.strip() for l in text.split('\n')[:20] if l.strip()]
    keywords = ['agreement', 'contract', 'license', 'deed', 'addendum', 'amendment']
    for line in lines:
        if any(k in line.lower() for k in keywords) and len(line) < 150:
            return line
    return lines[0] if lines else None

def extract_date_rule(text):
    patterns = [
        r'(?:dated?|as of|effective)\s+([A-Za-z]+ \d{1,2},?\s*\d{4})',
        r'(?:dated?|as of|effective)\s+(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        r'(\d{1,2}(?:st|nd|rd|th)?\s+day of\s+[A-Za-z]+,?\s*\d{4})',
    ]
    for pat in patterns:
        m = re.search(pat, text[:5000], re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None

print('✅ Rule-based extractors ready')

✅ Rule-based extractors ready


In [21]:
# CELL 7 — QA extractor with confidence threshold

def extract_qa(question, context):
    inputs = qa_tokenizer(
        question, context,
        return_tensors='pt',
        truncation=True,
        max_length=384
    ).to(device)

    with torch.no_grad():
        out = qa_model(**inputs)

    start_logits = out.start_logits[0]
    end_logits   = out.end_logits[0]

    best_score = -1e9
    best_s, best_e = 0, 0
    for s in range(len(start_logits)):
        for e in range(s, min(s + 40, len(end_logits))):
            score = start_logits[s].item() + end_logits[e].item()
            if score > best_score:
                best_score = score
                best_s, best_e = s, e

    if best_score < 2.0:
        return None

    answer = qa_tokenizer.decode(
        inputs['input_ids'][0][best_s:best_e + 1],
        skip_special_tokens=True
    ).strip()

    if len(answer) < 3 or len(answer) > 300:
        return None
    if answer.lower() in {'the', 'a', 'an', 'and', 'or', 'of', 'is', 'are'}:
        return None

    return answer


# ── BATCH version: runs ALL remaining questions on ONE chunk in one pass ──
def extract_qa_batch(questions_dict, context):
    """
    questions_dict: {clause_name: question_text} for clauses not yet found.
    Returns: {clause_name: answer_or_None}
    Much faster than calling extract_qa() one-by-one per chunk.
    """
    results = {}
    for clause, question in questions_dict.items():
        results[clause] = extract_qa(question, context)
    return results

print('✅ QA extractor ready')

✅ QA extractor ready


In [22]:
# CELL 8 — Fast Chunker
# KEY SPEEDUP: larger chunks (500 words) + less overlap (20 words)
# → fewer chunks → fewer QA calls → much faster

def chunk_text(text, size=500, overlap=20):
    """
    size=500  : was 300 — bigger chunks = fewer total chunks
    overlap=20: was 50  — minimal overlap, just enough to catch boundary clauses
    """
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(' '.join(words[i:i + size]))
        i += size - overlap
    return chunks

print('✅ Fast chunker ready (size=500, overlap=20)')

✅ Fast chunker ready (size=500, overlap=20)


In [23]:
# CELL 9 — Fast Hybrid Analyzer
#
# KEY SPEEDUPS vs previous version:
#   1. Larger chunks → fewer chunks to scan
#   2. Only scans first 60% of document (clauses rarely at the end)
#   3. Stops as soon as all 15 clauses found
#   4. Rule-based covers 4 clauses instantly

def analyze(text):
    results = {}

    # PASS 1: Rule-based — instant
    results['Document Name']  = extract_doc_name_rule(text)
    results['Parties']        = extract_parties_rule(text)
    results['Governing Law']  = extract_governing_law_rule(text)
    results['Agreement Date'] = extract_date_rule(text)
    results = {k: v for k, v in results.items() if v}

    # PASS 2: QA model — only on first 60% of document
    chunks = chunk_text(text)
    cutoff = max(1, int(len(chunks) * 0.6))   # skip last 40% (boilerplate)
    chunks = chunks[:cutoff]

    for chunk in chunks:
        # only ask about clauses not yet found
        remaining = {
            k: v for k, v in clause_questions.items()
            if k not in results
        }
        if not remaining:
            break   # all clauses found — stop early

        batch_results = extract_qa_batch(remaining, chunk)
        for clause, ans in batch_results.items():
            if ans:
                results[clause] = ans

    return results

print('✅ Fast hybrid analyzer ready')

✅ Fast hybrid analyzer ready


In [24]:
# CELL 10 — Risk Engine

def risk_analysis(results):
    risks = []
    if 'Cap On Liability' not in results:
        risks.append('⚠️ HIGH RISK — No cap on liability found')
    if 'Uncapped Liability' in results:
        risks.append('⚠️ HIGH RISK — Uncapped liability explicitly present')
    if 'Termination for Convenience' not in results:
        risks.append('⚠️ MEDIUM RISK — No termination for convenience clause')
    if 'Governing Law' not in results:
        risks.append('⚠️ MEDIUM RISK — Governing law not specified')
    if 'Non-Compete' in results:
        risks.append('⚠️ NOTE — Non-compete clause present')
    if 'Exclusivity' in results:
        risks.append('⚠️ NOTE — Exclusivity restriction present')
    if not risks:
        risks.append('✅ No major risk flags detected')
    return risks

def full_analysis(text):
    clauses = analyze(text)
    risks   = risk_analysis(clauses)
    return {'clauses': clauses, 'risks': risks}

print('✅ Risk engine ready')

✅ Risk engine ready


In [25]:
# CELL 11 — PDF Text Extractor

def pdf_to_text(file_bytes_or_path):
    if isinstance(file_bytes_or_path, (bytes, bytearray)):
        doc = fitz.open(stream=file_bytes_or_path, filetype='pdf')
    else:
        doc = fitz.open(file_bytes_or_path)
    return '\n'.join(page.get_text() for page in doc)

print('✅ PDF extractor ready')

✅ PDF extractor ready


In [26]:
# CELL 12 — 🚀 Launch Gradio UI

def run_analysis(files):
    if not files:
        return None, '**No files uploaded.**'

    all_clauses = []
    risk_md     = ''

    for file in files:
        text     = pdf_to_text(file.name)
        res      = full_analysis(text)
        filename = os.path.basename(file.name)

        row = {'Contract': filename}
        row.update(res['clauses'])
        all_clauses.append(row)

        risk_md += f'### {filename}\n'
        for r in res['risks']:
            risk_md += f'- {r}\n'
        risk_md += '\n'

    df = pd.DataFrame(all_clauses).fillna('Not found')
    return df, risk_md


def run_qa(question, files):
    if not files or not question.strip():
        return 'Please upload a PDF and type a question.'

    text   = pdf_to_text(files[0].name)
    chunks = chunk_text(text)

    for chunk in chunks:
        ans = extract_qa(question, chunk)
        if ans:
            return f'**Answer:** {ans}'

    return '❌ No clear answer found in this contract.'


with gr.Blocks(title='Contract Intelligence Engine') as demo:

    gr.Markdown(
        '# ⚖️ Contract Intelligence Engine\n'
        'Upload commercial contracts · Extract key clauses · Flag risks · Ask questions'
    )

    with gr.Tab('📄 Clause Extraction & Risk'):
        gr.Markdown(
            'Upload one or more PDF contracts. '
            'The system extracts **15 clause types** and flags risks automatically.'
        )
        upload_box  = gr.File(
            label='Upload contracts (PDF)',
            file_count='multiple',
            file_types=['.pdf']
        )
        analyze_btn = gr.Button('🔍 Analyze Contracts', variant='primary')
        clause_table = gr.Dataframe(
            label='Extracted Clauses — one row per contract',
            wrap=True
        )
        risk_box = gr.Markdown(label='Risk Report')

        analyze_btn.click(
            fn=run_analysis,
            inputs=[upload_box],
            outputs=[clause_table, risk_box]
        )

    with gr.Tab('💬 Ask a Question'):
        gr.Markdown(
            'Upload a contract and ask any question about its obligations and rights.\n\n'
            '**Examples:** *Does this contract have a non-compete clause?* · '
            '*What is the governing law?* · *Can either party terminate without cause?*'
        )
        qa_upload    = gr.File(
            label='Upload a contract (PDF)',
            file_count='multiple',
            file_types=['.pdf']
        )
        question_box = gr.Textbox(
            label='Your question',
            placeholder='Does this contract have a non-compete clause?',
            lines=2
        )
        ask_btn = gr.Button('💬 Ask', variant='primary')
        qa_out  = gr.Markdown()

        ask_btn.click(
            fn=run_qa,
            inputs=[question_box, qa_upload],
            outputs=[qa_out]
        )

demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://937a03365c36437702.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
